# Simple Benchmark: 1D Acoustic Wave Animation

Adapted from the finite-difference acoustic 1D animation notebook in the Computational Geophysics short course.

This is the low-time benchmark. It runs a small 1D wave simulation, stores frames, and renders the result as a JavaScript/HTML animation.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

plt.rcParams["animation.embed_limit"] = 80

In [ ]:
def ricker_source(time, f0=25.0, t0=None):
    if t0 is None:
        t0 = 4.0 / f0
    tau = np.pi * f0 * (time - t0)
    return (1.0 - 2.0 * tau**2) * np.exp(-tau**2)


def simulate_1d(nx=700, nt=360, frame_stride=4):
    xmax = 10000.0
    c0 = 334.0
    dx = xmax / (nx - 1)
    dt = 0.001
    eps = (c0 * dt / dx) ** 2
    isrc = nx // 2

    p = np.zeros(nx)
    pold = np.zeros(nx)
    pnew = np.zeros(nx)
    x = np.linspace(0.0, xmax, nx)
    frames = []

    start = time.perf_counter()
    for it in range(nt):
        pnew[1:-1] = 2.0 * p[1:-1] - pold[1:-1] + eps * (p[2:] - 2.0 * p[1:-1] + p[:-2])
        pnew[isrc] += dt**2 * ricker_source(it * dt)

        pold, p, pnew = p, pnew, pold

        if it % frame_stride == 0:
            frames.append(p.copy())

    elapsed = time.perf_counter() - start
    return x, np.asarray(frames), elapsed

x, frames_1d, simulation_seconds = simulate_1d()
print(f"simulation_seconds={simulation_seconds:.3f}")
print(f"frames={len(frames_1d)}, samples_per_frame={frames_1d.shape[1]}")

In [ ]:
start = time.perf_counter()
fig, ax = plt.subplots(figsize=(8, 3.8))
line, = ax.plot(x, frames_1d[0], lw=1.6)
ax.set_xlim(x.min(), x.max())
ymax = max(1e-12, np.max(np.abs(frames_1d)))
ax.set_ylim(-1.1 * ymax, 1.1 * ymax)
ax.set_xlabel("x [m]")
ax.set_ylabel("pressure")
ax.set_title("1D acoustic finite-difference wavefield")
ax.grid(True, alpha=0.25)


def update(frame_index):
    line.set_ydata(frames_1d[frame_index])
    ax.set_title(f"1D acoustic finite-difference wavefield | frame {frame_index + 1}/{len(frames_1d)}")
    return (line,)

ani = animation.FuncAnimation(fig, update, frames=len(frames_1d), interval=40, blit=True)
plt.close(fig)
animation_creation_seconds = time.perf_counter() - start
print(f"animation_creation_seconds={animation_creation_seconds:.3f}")

In [ ]:
start = time.perf_counter()
html = ani.to_jshtml()
render_seconds = time.perf_counter() - start
print(f"jshtml_render_seconds={render_seconds:.3f}")
print(f"html_size_mb={len(html) / 1024 / 1024:.2f}")
display(HTML(html))